# 🦣 Boto3 Basics

This notebook demonstrates how to connect to a local S3-compatible storage (RustFS) using the `boto3` Python SDK. You'll learn to list buckets, upload/download objects, and navigate the S3 API.

In [ ]:
import boto3

# Configure the S3 client pointing to local RustFS
s3 = boto3.client(
    service_name='s3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='adminpassword',
    region_name='us-east-1',
    use_ssl=False,
)

print("Connected to RustFS S3!")

In [ ]:
# List all existing buckets (bronze, silver, gold created by docker-compose)
response = s3.list_buckets()
buckets = [b['Name'] for b in response['Buckets']]

print("Buckets found:")
for name in buckets:
    print(f"  - {name}")

In [ ]:
# Create a new bucket for this lab
s3.create_bucket(Bucket='lab1-demo')
print("Bucket 'lab1-demo' created successfully!")

In [ ]:
from io import StringIO

# Create a simple text file in memory and upload it
content = "Hello, RustFS S3! This object was uploaded via boto3."
s3.put_object(
    Bucket='lab1-demo',
    Key='hello-s3.txt',
    Body=content.encode('utf-8'),
)
print("Uploaded hello-s3.txt to lab1-demo bucket!")

In [ ]:
# List objects inside lab1-demo
response = s3.list_objects_v2(Bucket='lab1-demo')
objects = response.get('Contents', [])

print("Objects in lab1-demo:")
for obj in objects:
    print(f"  Key: {obj['Key']}  |  Size: {obj['Size']} bytes")

In [ ]:
# Download the object and print its contents
response = s3.get_object(Bucket='lab1-demo', Key='hello-s3.txt')
body = response['Body'].read().decode('utf-8')

print("Downloaded content:")
print(body)

In [ ]:
# Generate a pre-signed URL valid for 1 hour (3600 seconds)
url = s3.generate_presigned_url(
    ClientMethod='get_object',
    Params={'Bucket': 'lab1-demo', 'Key': 'hello-s3.txt'},
    ExpiresIn=3600,
)

print("Pre-signed URL:")
print(url)

In [ ]:
# Cleanup: delete object and bucket
s3.delete_object(Bucket='lab1-demo', Key='hello-s3.txt')
print("Deleted object hello-s3.txt")

s3.delete_bucket(Bucket='lab1-demo')
print("Deleted bucket lab1-demo")

print("Cleanup complete!")